# Run the black-box judge with the Gemini API

1. In the repo root run `python scripts/make_judge_bundle.py --with-labels` and upload `judge_bundle.zip` below.
2. Add your Gemini key as a Colab secret named `GEMINI_API_KEY` (key icon, left panel).
3. Run all cells. Results download as `judge_results.zip`; unzip it into the repo root so the files land in `results/tables/`.

About 800 calls per model (100 conversations x 4 turns x 2 passes). Free-tier rate limits apply, so the run is paced at 4 s per call (about 55 minutes per model) and retries on HTTP 429. Create the key at aistudio.google.com.

In [ ]:
import os, zipfile
from google.colab import files, userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
up = files.upload()
for name in up:
    zipfile.ZipFile(name).extractall(".")
!pip -q install pyyaml tqdm

In [ ]:
JUDGE_MODEL = "gemini-3.8-flash"   # check ai.google.dev/gemini-api/docs/models for current ids
ENDPOINT = "https://generativelanguage.googleapis.com/v1beta/openai/chat/completions"
# (data tag, output name). Tag "" is Qwen; the others need data/multi_turn_gen_<tag>/ in the zip.
RUNS = [("", "qwen2.5-3b"), ("smollm2-1.7b", "smollm2-1.7b"), ("olmo-2-1b", "olmo-2-1b")]

In [ ]:
import os
for tag, name in RUNS:
    folder = f"data/multi_turn_gen_{tag}" if tag else "data/multi_turn_gen"
    if not os.path.isdir(folder):
        print("skip", name, "(no", folder + ")"); continue
    tag_arg = f"--data-tag {tag}" if tag else ""
    !PYTHONPATH=src python scripts/run_judge.py {tag_arg} --judge-model {JUDGE_MODEL} --endpoint {ENDPOINT} --api-key-env GEMINI_API_KEY --min-interval 4 --out results/tables/judge_gemini_{name}.jsonl

In [ ]:
import pandas as pd, glob
for f in sorted(glob.glob("results/tables/judge_gemini_*.jsonl")):
    d = pd.read_json(f, lines=True)
    print(f, len(d), d.status.value_counts().to_dict())
!zip -qr judge_results.zip results/tables/judge_gemini_*.jsonl
files.download("judge_results.zip")